In [30]:
# imports

from dotenv import load_dotenv
from pydantic import BaseModel, Field
from agents import Agent, Runner, WebSearchTool, ModelSettings, trace
from messenger_solution import send_email_tool
import asyncio
from IPython.display import display, Markdown

load_dotenv(override=True)

True

In [20]:
# Constants
NUM_SEARCH_TERMS = 6
MODEL_NAME = "gpt-5.4-mini-2026-03-17"

## 1. Create Agents

In [21]:
# Planner agent
# Input: a query string.
# Processing: agent will generate 'n' number of search terms.
# Output: The agent gives a structured output object, which contains 5 search terms and a reason for each.



class search_term(BaseModel):
    reason: str = Field(description="short reason for this search term")
    term:   str = Field(description="The term for the websearch")

class planner_results(BaseModel):
    search_terms: list[search_term] = Field(description="A list of web searchs to perform to best answer the query.")

instructions_planner_agent = f"""
You are an assistant who writes search terms for websearch.
Your Task: Given a query, you generate {NUM_SEARCH_TERMS} search terms to best answer the query. 
"""

planner_agent = Agent(name = "Planner Agent", instructions=instructions_planner_agent, model= MODEL_NAME, output_type=planner_results)

In [34]:
# Planner agent test
"""
query = "what are the skills expected from a mid level Agentic AI engineer?"

response = await Runner.run(planner_agent, query)
results = response.final_output.search_terms
search_terms_output = [result.term for result in results]
search_terms_output
"""

'\nquery = "what are the skills expected from a mid level Agentic AI engineer?"\n\nresponse = await Runner.run(planner_agent, query)\nresults = response.final_output.search_terms\nsearch_terms_output = [result.term for result in results]\nsearch_terms_output\n'

In [5]:
# search agent

instructions_search_agent = f"""
You are an assistant who does websearch.
Your Task: Given a search term, you perform a web search using the WebSearchTool.
You respond with a summary of your results in 2-3 paragraphs in a maximum of 300 words, by capturing the main points and being succinct. 
"""
search_model_settings = ModelSettings(tool_choice="required")
search_agent = Agent(name = "Search Agent", instructions=instructions_search_agent, model= "gpt-5.4-mini-2026-03-17", tools=[WebSearchTool()], model_settings=search_model_settings)

In [33]:
# search agent test

#responses = await asyncio.gather(*[Runner.run(search_agent, term) for term in search_terms_output]) 

In [32]:
#summaries = ""
#for i, response in enumerate(responses):
#    summaries += f"Summary {i + 1}:\n\n{response.final_output}\n\n"

#print(summaries)

In [8]:
# writer agent

instructions_writer_agent = """
You are a senior researcher tasked with writing a cohesive report for a research query.
You will be provided with the original query, and some research.
Generate a comprehensive report based on the research and the query.
The final output should be in markdown format, and it should be lengthy and detailed. Aim 
for 5-10 pages of content, at least 1000 words.
"""


class ReportData(BaseModel):
    short_summary: str = Field(description="A short 2-3 sentence summary of the findings.")
    markdown_report: str = Field(description="The final report")
    follow_up_questions: list[str] = Field(description="Suggested topics to research further")


writer_agent = Agent(name = "Writer Agent", instructions=instructions_writer_agent, model= "gpt-5.6", output_type=ReportData)

In [ ]:
#writer_response = await Runner.run(writer_agent, query + "\n\n" + summaries)
#response = writer_response.final_output
#display(Markdown(response.markdown_report))
#display(Markdown(response.short_summary))

A mid-level Agentic AI engineer is expected to turn LLM capabilities into dependable software systems that can reason, retrieve context, call tools, manage state, and complete multi-step tasks. The role combines backend engineering, agent/workflow design, RAG and evaluation, cloud operations, security, and product judgment; shipping reliable production systems matters more than expertise in any single framework.

In [19]:
# email agent

instructions_email_agent = """
You are provided with a detailed report. Use your tool to send an email, converting the report into
a clean, well presented HTML email with an appropriate subject line.
"""

email_sender_agent = Agent(name = "Email Sender Agent", instructions=instructions_email_agent, model= "gpt-5.4-mini-2026-03-17", tools=[send_email_tool], model_settings=ModelSettings(tool_choice="required"))

## 2. Orchestrate the workflow

In [24]:
# Orchestration

async def run_searches(query: str):
    print("Planning searches...")
    result = await Runner.run(planner_agent, f"Query: {query}")
    searches = result.final_output.search_terms
    print(f"Will perform {len(searches)} searches")
    tasks = [search(item) for item in searches]
    results = await asyncio.gather(*tasks)
    print("Finished searching")
    return results

async def search(item: search_term):
    input_message = f"Search term: {item.term}\nReason for search: {item.reason}"
    result = await Runner.run(search_agent, input_message)
    return result.final_output

['For Bavaria specifically, the strongest finance/business options appear to be **Technical University of Munich (TUM)**, **LMU Munich**, and **Munich Business School (MBS)**, with **University of Passau** also showing up well in subject and student-satisfaction rankings. TUM School of Management says it is ranked **#1 business school in Germany** and **#1 in the EU** in the 2025 Times Higher Education business-school ranking, and its Master’s in Finance and Information Management placed **31st globally** in the QS Business Masters Rankings. LMU Munich’s School of Management reports being **#1 for Business Administration in Germany** and highly placed in finance-related subject rankings. ([mgt.tum.de](https://www.mgt.tum.de/?utm_source=openai))\n\nFor a broader “reputation” view, private business schools in Bavaria like **Munich Business School** are often described as among the leading private business schools in the German-speaking region, while **University of Passau** is reported a

In [28]:
async def write_report(query: str, search_results: list[str]):
    print("Writing the report...")
    input_message = f"User query: {query} \n\n {search_results}"
    result = await Runner.run(writer_agent, input_message)
    print("Finished writing report")
    return result.final_output

async def send_report_email(report: ReportData):
    print("Writing email...")
    result = await Runner.run(email_sender_agent, report.markdown_report)
    print("Email sent")
    return result.final_output

In [31]:
query = "Generate a report, that explains about conccurency in python, and gives some coding excercises"

with trace("Research workflow"):
    print("Starting research...")
    search_results = await run_searches(query)
    report         = await write_report(query, search_results)
    await send_report_email(report)
    print("Woohooo")

Starting research...
Planning searches...
Will perform 6 searches
Finished searching
Writing the report...
Finished writing report
Writing email...
Email sent
Woohooo
